# Step Goal Early-Warning Predictor* - Modeling

## 1. Problem Statement

**Problem:** I have a step goal of 10k per day. After a long afternoon, I've gotten halfway through that goal. But when I reach home, all I want to do is chill on my couch or just take a nap, then things get busy and I end up staying halfway through that goal for the rest of the day.

**Objective:** What this project aims to do is to predict, based on your current steps taken and time of day, whether you'll be able to reach that goal or not. If not, it'll give advice like "Take a 10 minute walk around your campus" so that you have the best chance of reaching the goal in time. As a secondary feature, another model would estimate daily calories burned from step count, so the user can see how more steps relate to total energy expenditure. The demo can show it next to the advice.

**Why it matters:** When I get home I tend to skip a walk, and the shortfall only shows up when it’s too late to fix. An early warning turns that into something I can still act on.

**Expected outcome:** A trained classifier, a calorie regression, an evaluation, and a working frontend that serves a clear purpose. My classifier should beat a simple baseline.

## 2. Loading & Filtering Data

### 2.1 Load Processed Files

In [1]:
import pandas as pd

In [2]:
minute = pd.read_csv('./data/processed_minute_steps.csv', parse_dates=['ActivityMinute'], date_format='%m/%d/%Y %I:%M:%S %p')
daily = pd.read_csv('./data/processed_daily.csv', parse_dates=['ActivityDate'], date_format='%Y-%m-%d')
# Both files come from 01_eda_cleaning.ipynb

In [3]:
minute.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1325580 entries, 0 to 1325579
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   Id              1325580 non-null  int64         
 1   ActivityMinute  1325580 non-null  datetime64[ns]
 2   Steps           1325580 non-null  int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 30.3 MB


In [4]:
daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 934 entries, 0 to 933
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Id                        934 non-null    int64         
 1   ActivityDate              934 non-null    datetime64[ns]
 2   Steps                     934 non-null    int64         
 3   TotalSteps                934 non-null    int64         
 4   TotalDistance             934 non-null    float64       
 5   TrackerDistance           934 non-null    float64       
 6   LoggedActivitiesDistance  934 non-null    float64       
 7   VeryActiveDistance        934 non-null    float64       
 8   ModeratelyActiveDistance  934 non-null    float64       
 9   LightActiveDistance       934 non-null    float64       
 10  SedentaryActiveDistance   934 non-null    float64       
 11  VeryActiveMinutes         934 non-null    int64         
 12  FairlyActiveMinutes   

### 2.2 Remove High-Mismatch Users

Before I train my model on data that's present in two different files, I need to make sure that the data I'm using is consistent across files. So I'll need to determine which users have a high disagreement between their daily and minute-level step counts, and drop them before I train.

I'll take some code from 01_eda_cleaning, with some changes to include users with zero mismatches instead of just showing NaN:

In [5]:
mismatches = daily[daily['StepsDifference'] != 0] # Only the rows with a complete agreement

In [6]:
total_days = daily.groupby('Id').size() # The total number of days for each Id
mismatch_counts = mismatches['Id'].value_counts() # The number of rows that disagree, for each Id

# Calculate the percentage of disagreement:
mismatch_rate = round(((mismatch_counts.reindex(total_days.index, fill_value=0) / total_days) * 100), 1).sort_values(ascending=False)
mismatch_rate

Id
8877689391    83.9
8583815059    50.0
4020332650    48.4
7086361926    35.5
4319703577    29.0
4057192912    25.0
1503960366    23.3
2320127002    22.6
1644430081    20.0
3977333714    17.2
1624580081    16.1
4702921684    16.1
4445114986    16.1
5577150313    13.3
4558609924    12.9
8053475328    12.9
5553957443    12.9
6775888955    11.5
6117666160    10.7
2026352035     9.7
4388161847     9.7
8378563200     9.7
1844505072     6.5
2873212765     6.5
2347167796     5.6
8253242879     5.6
3372868164     5.0
6290855005     3.6
8792009665     3.6
6962181067     3.2
2022484408     3.2
1927972279     0.0
7007744171     0.0
dtype: float64

I will drop users whose minute-level and daily steps disagreed on more than 40% of their days, which is where the data shows a clear separation. There's no clear break below the top 3, and dropping more users costs me data I don't have much of, so 40% should be a reasonable threshold.

In [7]:
drop_list = mismatch_rate[mismatch_rate > 40].index
drop_list

Index([8877689391, 8583815059, 4020332650], dtype='int64', name='Id')

In [8]:
keep_daily = ~daily['Id'].isin(drop_list)
daily_cleaned = daily[keep_daily].copy()

In [9]:
keep_minute = ~minute['Id'].isin(drop_list)
minute_cleaned = minute[keep_minute].copy()

In [10]:
daily_cleaned.shape

(842, 17)

In [11]:
minute_cleaned.shape

(1194540, 3)

This process removed 3 of 33 users, leaving 30.

**Daily rows:** 934 -> 842 (92 rows dropped)

**Minute rows:** 1,325,580 -> 1,194,540 *(131,040 rows dropped)*

### 2.3 Keep Complete Days Only

Since the model predicts whether you'll reach the goal by the end of the day or not, it should only train on complete days, or else it could give a false prediction like "you'll reach the goal by today" if all it has to base that prediction off is an incomplete day.

In [12]:
minute_cleaned['ActivityDate'] = minute_cleaned['ActivityMinute'].dt.normalize()

In [13]:
# This will be useful during model training. More on this later
# minute_cleaned['Hour'] = minute_cleaned['ActivityMinute'].dt.hour

In [14]:
minutes_per_day = minute_cleaned.groupby(['Id', 'ActivityDate']).size()
minutes_per_day # Gets the total amount of minutes per day for each user

Id          ActivityDate
1503960366  2016-04-12      1440
            2016-04-13      1440
            2016-04-14      1440
            2016-04-15      1440
            2016-04-16      1440
                            ... 
8792009665  2016-05-05      1440
            2016-05-06      1440
            2016-05-07      1440
            2016-05-08      1440
            2016-05-09      1440
Length: 842, dtype: int64

In [15]:
minutes_per_day.value_counts() # A "bird's eye" view of how many rows are actually full days

1440    814
960       7
720       4
900       4
780       3
660       2
600       2
1260      1
360       1
240       1
1020      1
60        1
1380      1
Name: count, dtype: int64

814 user-days out of 842 are complete, with 28 days being the partial ones. I'll only be losing ~3% of data when I drop them, which isn't a lot.

In [16]:
# Since .size() only returns a collapsed version of my data, I need to use .transform('size') instead
# so I have a mask that's exactly 1,194,540 rows long, which can be used as a mask.
total_minutes_per_day = minute_cleaned.groupby(['Id', 'ActivityDate'])['ActivityDate'].transform('size')
total_minutes_per_day.value_counts()

ActivityDate
1440    1172160
960        6720
900        3600
720        2880
780        2340
1380       1380
660        1320
1260       1260
600        1200
1020       1020
360         360
240         240
60           60
Name: count, dtype: int64

In [17]:
minute_cleaned = minute_cleaned[total_minutes_per_day == 1440]
minute_cleaned.groupby(['Id', 'ActivityDate']).size().value_counts()

1440    814
Name: count, dtype: int64

Doing it for the daily dataset too, so both tables describe the same user-days

In [22]:
complete_days = minute_cleaned[['Id', 'ActivityDate']].drop_duplicates()
complete_days.shape

(814, 2)

In [23]:
daily_cleaned = pd.merge(daily_cleaned, complete_days, on=['Id', 'ActivityDate'])

In [24]:
len(daily_cleaned)

814

In [25]:
daily_cleaned['Id'].nunique()

30

In [26]:
len(minute_cleaned)

1172160

## 3. Defining the Prediction Problem

### 3.1 Structure of the Training Table

Here's what the structure of the training table would look like:

| Id | ActivityDate | MinuteOfDay | StepsSoFar | FinalTotal | Reached10k |
|----------|----------|----------|----------|----------|----------|
| A | Apr 20 | 600 | 1,200 | 11,400 | 1 |
| A | Apr 20 | 870 | 3,000 | 11,400 | 1 |
| A | Apr 21 | 1,110 | 700 | 4,300 | 0 |



*Illustrative values (600 = 10:00am, 870 = 2:30pm, 1,110 = 6:30pm)

We'll call this table `checkpoints`. The `checkpoints` table will be built from the minute-level step data. Each row contains a user, a day, and a time of day measured in minutes since midnight. Then we have the amount of steps taken so far, and our target column, which tells us whether that user reached 10k steps or not by the end of the day.

Each day produces one row per checkpoint, every 30 minutes from 8:00am to 9:30pm, so the model sees what “steps so far” looks like at different times of day, and the app needs that because the user can ask at any time.

To prevent leakage: “steps so far” at a time like 3pm must count only minutes *before* 3pm (as far as an actual person could see at that point). If any later minute leaks in, that means the model is peeking at the answer, and it will score very well on the test set but be useless in an actual prediction because it's not actually predicting anything. The checkpoint minute itself shouldn’t be counted (at 3:00pm it'll count up to 2:59pm).

Moreover, `FinalTotal` and `Reached10k` will never be used as features.

### 3.2 Target Variable

The target column is `Reached10k`: 1 if the user’s full-day step total was 10,000 or more, otherwise it's 0. The full-day total is the sum of that day’s minute-level steps (the same source as the features), and only complete days are used (done in 2.3: 814 kept, 28 dropped), so a partial day can’t produce a false 0. I've picked 10k steps as a fixed goal as that's what the project is built around.

`FinalTotal` is stored in the table next to the target.

All of a day’s checkpoints share the same value, since the outcome belongs to the day, not the checkpoint.

## 4. Feature Engineering

In [28]:
minute_cleaned = minute_cleaned.sort_values(by=['Id', 'ActivityMinute'])

In [47]:
minute_cleaned['MinuteOfDay'] = minute_cleaned['ActivityMinute'].dt.hour * 60 + minute_cleaned['ActivityMinute'].dt.minute

In [48]:
# This gets the total of steps per user-day, but then subtracting it with the steps taken
# in the current minute, which gives us our "steps so far" - steps taken strictly before this minute
minute_cleaned['StepsSoFar'] = minute_cleaned.groupby(['Id', 'ActivityDate'])['Steps'].cumsum() - minute_cleaned['Steps']

In [44]:
minute_cleaned['FinalTotal'] = minute_cleaned.groupby(['Id', 'ActivityDate'])['Steps'].transform('sum')

In [53]:
minute_cleaned['Reached10k'] = (minute_cleaned['FinalTotal'] >= 10000).astype(int) # Converts boolean to int

In [56]:
checkpoints = minute_cleaned[(minute_cleaned['MinuteOfDay'] % 30 == 0) & (minute_cleaned['MinuteOfDay'].between(480, 1290))]

In [58]:
checkpoints.head()

,Id,ActivityMinute,Steps,ActivityDate,Hour,MinuteOfDay,StepsSoFar,FinalTotal,Reached10k
480,1503960366,2016-04-12 08:00:00,0,2016-04-12,8,480,684,13158,1
510,1503960366,2016-04-12 08:30:00,0,2016-04-12,8,510,684,13158,1
540,1503960366,2016-04-12 09:00:00,39,2016-04-12,9,540,934,13158,1
570,1503960366,2016-04-12 09:30:00,0,2016-04-12,9,570,1922,13158,1
600,1503960366,2016-04-12 10:00:00,0,2016-04-12,10,600,2798,13158,1


In [64]:
checkpoints.drop(['ActivityDate', 'Steps'], axis=1, inplace=True)

KeyError: "['ActivityDate', 'Steps'] not found in axis"

In [65]:
checkpoints

,Id,ActivityMinute,MinuteOfDay,StepsSoFar,FinalTotal,Reached10k
480,1503960366,2016-04-12 08:00:00,480,684,13158,1
510,1503960366,2016-04-12 08:30:00,510,684,13158,1
540,1503960366,2016-04-12 09:00:00,540,934,13158,1
570,1503960366,2016-04-12 09:30:00,570,1922,13158,1
600,1503960366,2016-04-12 10:00:00,600,2798,13158,1
...,...,...,...,...,...,...
1281270,8792009665,2016-05-09 19:30:00,1170,0,0,0
1281300,8792009665,2016-05-09 20:00:00,1200,0,0,0
1281330,8792009665,2016-05-09 20:30:00,1230,0,0,0
1281360,8792009665,2016-05-09 21:00:00,1260,0,0,0


In [66]:
checkpoints.Reached10k.mean()

np.float64(0.3230958230958231)